# Train DL Classifier



In [1]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, average_precision_score, precision_recall_fscore_support
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

In [2]:
ROOT = Path.cwd()
if not ((ROOT / "work").exists() and (ROOT / "baseline").exists()):
    candidate = ROOT / "GRACE-java"
    if (candidate / "work").exists() and (candidate / "baseline").exists():
        ROOT = candidate
    else:
        raise FileNotFoundError("Cannot locate GRACE-java root. Open notebook from repo root or GRACE-java.")

WORK_DIR = ROOT / "work"
MODEL_DIR = WORK_DIR / "dl_classifier"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "model_name": "microsoft/graphcodebert-base",
    "max_length": 512,
    "code_token_budget": 384,
    "ast_token_budget": 96,
    "batch_size": 4,
    "grad_accum_steps": 2,
    "epochs": 5,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout": 0.1,
    "threshold_strategy": "best_f1",
    "target_recall": 0.95,
    "checkpoint_metric": "val_ap",
    "seed": 42,
    "gradient_clip": 1.0,
    "code_max_chars": 12000,
    "ast_max_tokens": 256,
    "rebalance_ratio_threshold": 1.5,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"root": str(ROOT), "device": str(DEVICE)})

{'root': 'c:\\Users\\Admin\\Documents\\1. UET\\lab\\VulGuardVN\\GRACE-java', 'device': 'cuda'}


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def rows_to_frame(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    frame["code"] = frame["code"].fillna("").astype(str)
    if "ast_seq" not in frame:
        frame["ast_seq"] = ""
    frame["ast_seq"] = frame["ast_seq"].fillna("").astype(str)
    frame["label"] = frame["label"].astype(int)
    frame["sample_id"] = frame["sample_id"].astype(str)
    return frame


def validate_binary_split(frame: pd.DataFrame, split_name: str) -> None:
    counts = frame["label"].value_counts().to_dict()
    if len(counts) < 2:
        raise ValueError(
            f"{split_name} split has a single class: {counts}. "
            "Rebuild splits with baseline/01_split_dataset.py before training."
        )


def encode_code_ast(tokenizer, code: str, ast_seq: str, max_length: int) -> dict[str, torch.Tensor]:
    code = (code or "").strip()[: CONFIG["code_max_chars"]]
    ast_seq = " ".join((ast_seq or "").split()[: CONFIG["ast_max_tokens"]])

    ast_text = f"AST {ast_seq}" if ast_seq else ""
    ast_ids = tokenizer.encode(
        ast_text,
        add_special_tokens=False,
        truncation=True,
        max_length=CONFIG["ast_token_budget"],
    )
    has_pair = bool(ast_ids)
    special_tokens = tokenizer.num_special_tokens_to_add(pair=has_pair)
    code_budget = min(CONFIG["code_token_budget"], max_length - special_tokens - len(ast_ids))
    code_budget = max(1, code_budget)
    code_ids = tokenizer.encode(
        code,
        add_special_tokens=False,
        truncation=True,
        max_length=code_budget,
    )

    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id or 1
    if bos_id is None:
        bos_id = tokenizer.cls_token_id
    if eos_id is None:
        eos_id = tokenizer.sep_token_id
    if bos_id is None or eos_id is None:
        raise ValueError("Tokenizer is missing bos/eos token ids needed to build pair inputs.")

    if has_pair:
        input_ids = [bos_id] + code_ids + [eos_id, eos_id] + ast_ids + [eos_id]
    else:
        input_ids = [bos_id] + code_ids + [eos_id]
    attention_mask = [1] * len(input_ids)
    pad_len = max_length - len(input_ids)
    if pad_len < 0:
        input_ids = input_ids[:max_length]
        attention_mask = attention_mask[:max_length]
    else:
        input_ids += [pad_id] * pad_len
        attention_mask += [0] * pad_len

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


def compute_binary_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float) -> dict:
    preds = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "positive_rate": float(preds.mean()) if len(preds) else 0.0,
    }


def select_threshold(labels: np.ndarray, probs: np.ndarray, strategy: str, target_recall: float) -> dict:
    candidates = np.unique(np.round(np.concatenate([np.linspace(0.01, 0.99, 99), probs]), 4))
    scored = [compute_binary_metrics(labels, probs, float(t)) for t in candidates]
    if strategy == "best_f1":
        best = max(scored, key=lambda m: (m["f1"], m["precision"], m["recall"]))
        best["mode"] = "best_f1"
        return best
    feasible = [m for m in scored if m["recall"] >= target_recall]
    if feasible:
        best = max(feasible, key=lambda m: (m["f1"], m["precision"], -m["positive_rate"]))
        best["mode"] = "target_recall"
        return best
    best = max(scored, key=lambda m: (m["f1"], m["recall"], m["precision"]))
    best["mode"] = "max_f1"
    return best

In [4]:
class CodeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.frame = frame.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.frame.iloc[idx]
        enc = encode_code_ast(self.tokenizer, row.code, row.ast_seq, self.max_length)
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": torch.tensor(float(row.label), dtype=torch.float32),
        }


class GraphCodeBERTClassifier(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls)).squeeze(-1)


def make_train_sampler(frame: pd.DataFrame) -> WeightedRandomSampler:
    label_counts = frame["label"].value_counts().to_dict()
    weights = frame["label"].map(lambda label: 1.0 / label_counts[int(label)]).to_numpy(dtype=np.float64)
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)


def should_rebalance(frame: pd.DataFrame) -> bool:
    counts = frame["label"].value_counts().to_dict()
    if len(counts) < 2:
        return False
    minority = min(counts.values())
    majority = max(counts.values())
    return (majority / max(minority, 1)) >= CONFIG["rebalance_ratio_threshold"]


@torch.no_grad()
def predict_probs(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    probs = []
    for batch in tqdm(loader, desc="infer", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
    return np.asarray(probs, dtype=np.float32)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    criterion: nn.Module,
) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        raw_loss = criterion(logits, labels)
        loss = raw_loss / CONFIG["grad_accum_steps"]
        loss.backward()
        total_loss += float(raw_loss.item())

        if step % CONFIG["grad_accum_steps"] == 0 or step == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    return total_loss / max(len(loader), 1)


def attach_scores(frame: pd.DataFrame, probs: np.ndarray, threshold: float) -> list[dict]:
    rows = []
    for row, prob in zip(frame.to_dict(orient="records"), probs.tolist()):
        row["dl_prob"] = float(prob)
        row["dl_pred"] = int(prob >= threshold)
        row["route_to_llm"] = bool(prob >= threshold)
        rows.append(row)
    return rows

## Load Data

In [5]:
set_seed(CONFIG["seed"])

train_df = rows_to_frame(load_jsonl(WORK_DIR / "train_ast.jsonl"))
val_df = rows_to_frame(load_jsonl(WORK_DIR / "val_ast.jsonl"))
test_df = rows_to_frame(load_jsonl(WORK_DIR / "test_ast.jsonl"))

for split_name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    validate_binary_split(frame, split_name)

summary_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "positive": [int(train_df.label.sum()), int(val_df.label.sum()), int(test_df.label.sum())],
    }
)
summary_df["negative"] = summary_df["rows"] - summary_df["positive"]
summary_df

,split,rows,positive,negative
0,train,1626,768,858
1,val,124,59,65
2,test,105,45,60


## Build Tokenizer, Loader, Model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

train_ds = CodeDataset(train_df, tokenizer, CONFIG["max_length"])
val_ds = CodeDataset(val_df, tokenizer, CONFIG["max_length"])
test_ds = CodeDataset(test_df, tokenizer, CONFIG["max_length"])

rebalance = should_rebalance(train_df)
train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    sampler=make_train_sampler(train_df) if rebalance else None,
    shuffle=not rebalance,
)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

model = GraphCodeBERTClassifier(CONFIG["model_name"], CONFIG["dropout"]).to(DEVICE)
pos = float(train_df["label"].sum())
neg = float(len(train_df) - pos)
pos_weight_value = max(1.0, neg / max(pos, 1.0)) if rebalance else 1.0
pos_weight = torch.tensor(pos_weight_value, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
updates_per_epoch = math.ceil(len(train_loader) / CONFIG["grad_accum_steps"])
total_steps = updates_per_epoch * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

setup_info = {
    "device": str(DEVICE),
    "pos_weight": float(pos_weight.item()),
    "rebalance": rebalance,
    "train_steps": total_steps,
    "warmup_steps": warmup_steps,
}
setup_info

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'device': 'cuda',
 'pos_weight': 1.0,
 'rebalance': False,
 'train_steps': 1020,
 'warmup_steps': 102}

## Train

In [7]:
history = []
best = {"score": -math.inf, "epoch": 0, "threshold": 0.5}
best_state = None

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion)
    val_probs = predict_probs(model, val_loader)
    val_labels = val_df["label"].to_numpy(dtype=np.int64)
    threshold_info = select_threshold(
        val_labels,
        val_probs,
        CONFIG["threshold_strategy"],
        CONFIG["target_recall"],
    )
    ap = float(average_precision_score(val_labels, val_probs))
    row = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "val_ap": ap,
        **threshold_info,
    }
    history.append(row)
    print(row)

    score = row[CONFIG["checkpoint_metric"]] if CONFIG["checkpoint_metric"] in row else row["val_ap"]
    if score > best["score"]:
        best = {
            "score": float(score),
            "epoch": epoch,
            "threshold": row["threshold"],
            "mode": row["mode"],
            "metric": CONFIG["checkpoint_metric"],
        }
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is None:
    raise RuntimeError("No checkpoint captured during training.")

pd.DataFrame(history)

train:   0%|          | 0/407 [00:00<?, ?it/s]

infer:   0%|          | 0/31 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.7088781813438753, 'val_ap': 0.5332582820954213, 'threshold': 0.4985, 'accuracy': 0.5403225806451613, 'precision': 0.509090909090909, 'recall': 0.9491525423728814, 'f1': 0.6627218934911243, 'positive_rate': 0.8870967741935484, 'mode': 'best_f1'}


train:   0%|          | 0/407 [00:00<?, ?it/s]

infer:   0%|          | 0/31 [00:00<?, ?it/s]

...

{'epoch': 4, 'train_loss': 0.6466150664214128, 'val_ap': 0.5163450364402553, 'threshold': 0.45, 'accuracy': 0.5241935483870968, 'precision': 0.5, 'recall': 0.9830508474576272, 'f1': 0.6628571428571428, 'positive_rate': 0.9354838709677419, 'mode': 'best_f1'}


train:   0%|          | 0/407 [00:00<?, ?it/s]

infer:   0%|          | 0/31 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.6229936740834824, 'val_ap': 0.5199369285462208, 'threshold': 0.3964, 'accuracy': 0.5241935483870968, 'precision': 0.5, 'recall': 0.9830508474576272, 'f1': 0.6628571428571428, 'positive_rate': 0.9354838709677419, 'mode': 'best_f1'}


,epoch,train_loss,val_ap,threshold,accuracy,precision,recall,f1,positive_rate,mode
0,1,0.708878,0.533258,0.4985,0.540323,0.509091,0.949153,0.662722,0.887097,best_f1
1,2,0.695946,0.530524,0.4961,0.548387,0.514286,0.915254,0.658537,0.846774,best_f1
2,3,0.670505,0.519149,0.3633,0.516129,0.495726,0.983051,0.659091,0.943548,best_f1
3,4,0.646615,0.516345,0.4500,0.524194,0.500000,0.983051,0.662857,0.935484,best_f1
4,5,0.622994,0.519937,0.3964,0.524194,0.500000,0.983051,0.662857,0.935484,best_f1


## Save Best Checkpoint And Evaluate

In [8]:
checkpoint = {
    "state_dict": best_state,
    "config": CONFIG,
    "model_name": CONFIG["model_name"],
}
torch.save(checkpoint, MODEL_DIR / "best_model.pt")
tokenizer.save_pretrained(MODEL_DIR / "tokenizer")
(MODEL_DIR / "history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
(MODEL_DIR / "threshold.json").write_text(json.dumps(best, ensure_ascii=False, indent=2), encoding="utf-8")

best_checkpoint = torch.load(MODEL_DIR / "best_model.pt", map_location=DEVICE)
model = GraphCodeBERTClassifier(best_checkpoint["model_name"], best_checkpoint["config"]["dropout"]).to(DEVICE)
model.load_state_dict(best_checkpoint["state_dict"])
threshold = float(json.loads((MODEL_DIR / "threshold.json").read_text(encoding="utf-8"))["threshold"])

val_probs = predict_probs(model, val_loader)
test_probs = predict_probs(model, test_loader)

val_metrics = compute_binary_metrics(val_df["label"].to_numpy(dtype=np.int64), val_probs, threshold)
test_metrics = compute_binary_metrics(test_df["label"].to_numpy(dtype=np.int64), test_probs, threshold)
print("val", val_metrics)
print("test", test_metrics)

val_rows = attach_scores(val_df, val_probs, threshold)
test_rows = attach_scores(test_df, test_probs, threshold)
write_jsonl(MODEL_DIR / "val_scored.jsonl", val_rows)
write_jsonl(MODEL_DIR / "test_scored.jsonl", test_rows)

metrics_summary = {
    "threshold": threshold,
    "split_summary": summary_df.to_dict(orient="records"),
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "val_route_rate": float(np.mean([row["route_to_llm"] for row in val_rows])),
    "test_route_rate": float(np.mean([row["route_to_llm"] for row in test_rows])),
}
(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding="utf-8")
metrics_summary

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


infer:   0%|          | 0/31 [00:00<?, ?it/s]

infer:   0%|          | 0/27 [00:00<?, ?it/s]

val {'threshold': 0.4985, 'accuracy': 0.5403225806451613, 'precision': 0.509090909090909, 'recall': 0.9491525423728814, 'f1': 0.6627218934911243, 'positive_rate': 0.8870967741935484}
test {'threshold': 0.4985, 'accuracy': 0.42857142857142855, 'precision': 0.422680412371134, 'recall': 0.9111111111111111, 'f1': 0.5774647887323944, 'positive_rate': 0.9238095238095239}


{'threshold': 0.4985,
 'split_summary': [{'split': 'train',
   'rows': 1626,
   'positive': 768,
   'negative': 858},
  {'split': 'val', 'rows': 124, 'positive': 59, 'negative': 65},
  {'split': 'test', 'rows': 105, 'positive': 45, 'negative': 60}],
 'val_metrics': {'threshold': 0.4985,
  'accuracy': 0.5403225806451613,
  'precision': 0.509090909090909,
  'recall': 0.9491525423728814,
  'f1': 0.6627218934911243,
  'positive_rate': 0.8870967741935484},
 'test_metrics': {'threshold': 0.4985,
  'accuracy': 0.42857142857142855,
  'precision': 0.422680412371134,
  'recall': 0.9111111111111111,
  'f1': 0.5774647887323944,
  'positive_rate': 0.9238095238095239},
 'val_route_rate': 0.8870967741935484,
 'test_route_rate': 0.9238095238095239}